# PicoCal — Transformer tuning (notebook 06)

## Config

Every knob in one place. Increase `D_MODEL`, `LAYERS`, `WINDOW`, or `EPOCHS` to push harder.

In [1]:
import sys
from pathlib import Path
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, split, resolution, collate, TokenDS, EPS

FILES = 100
WINDOW = 3
REGION = 3
VERTEX = 100.0
ALL_REGIONS = True
D_MODEL = 96
NHEAD = 4
LAYERS = 3
DROPOUT = 0.1
EPOCHS = 150
PATIENCE = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
BATCH = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
{"device": DEVICE, "window": WINDOW, "d_model": D_MODEL, "layers": LAYERS, "epochs": EPOCHS}

{'device': 'cuda', 'window': 3, 'd_model': 96, 'layers': 3, 'epochs': 150}

## Build the dataset

Reuse the same builder (vertex cut, seed window, tokens). One token = one cell, F=12. We always test on the held-out R3 set; with `ALL_REGIONS=True` the transformer trains on every region (more data), which notebook 05 found to be the best config. The BDT / total_energy bar stays trained on R3 only, so the comparison is on the identical R3 test set.

In [2]:
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:FILES]
D = build(files, WINDOW, VERTEX)
y = D["y"]; Et = D["Etrue"]; toks = D["tok_seed"]
ridx = np.flatnonzero(D["region"] == REGION)
rtr, rva, rte = (ridx[s] for s in split(len(ridx)))
ttr = np.setdiff1d(np.setdiff1d(np.arange(len(y)), rte), rva) if ALL_REGIONS else rtr
{"total_clusters": len(y), "R3_clusters": len(ridx), "transformer_train": len(ttr),
 "R3_val": len(rva), "R3_test": len(rte), "token_dim": toks[int(ridx[0])].shape[1]}

{'total_clusters': 36852,
 'R3_clusters': 10692,
 'transformer_train': 33644,
 'R3_val': 1604,
 'R3_test': 1604,
 'token_dim': 12}

## Normalize the continuous features

Fit mean/std on the **training** cells only (first 7 columns), so the test set never leaks into normalization.

In [3]:
cont = np.concatenate([toks[i][:, :7] for i in ttr], 0)
mean = cont.mean(0); std = cont.std(0) + EPS
def loader(idx, shuffle):
    return DataLoader(TokenDS([toks[i] for i in idx], y[idx], mean, std, 7),
                      batch_size=BATCH, shuffle=shuffle, collate_fn=collate)
mean.round(2)

array([6.44, 4.32, 6.15, 0.  , 0.  , 1.13, 3.56], dtype=float32)

## Architecture and references

A permutation-invariant set transformer, following the 2022-2026 calorimeter literature (see `reports/architecture-references.md`):

| design choice | reference |
|---|---|
| ~12 raw features per cell embedded to `d_model` (~64) | CLAS12, arXiv:2503.11277 (2025) — 17 features -> 64-dim |
| set tokens + masked mean pool (position is a feature, not an index) | Deep Sets (2017), Set Transformer (2019) |
| local `rel/pitch` geometry | ClusTEX, arXiv:2603.18172 (2026) — local + global coordinates |
| attention over cells | ParT (2022) + our E11 ablation (notebook 05) |
| trained from scratch on simulation | Object Condensation + GravNet, Kieseler 2022 |

Upgrade path (not yet here): a global-position channel (ClusTEX) or GravNet-learned PE (CLAS12), and a ParT-style pairwise interaction bias. The transformer's edge over clustering is documented mainly for overlapping / boosted showers, matching notebook 01 where the clean single-photon case is total-energy-dominated.

## The tuned transformer

Bigger than before: `D_MODEL=96`, 3 layers, dropout in both the encoder and the head. Still permutation-invariant with masked mean pooling.

In [4]:
class TunedTransformer(nn.Module):
    def __init__(self, in_dim, d_model, nhead, layers, dropout):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=4 * d_model,
                                           dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(d_model, 1))

    def forward(self, x, m):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        w = m.unsqueeze(-1).float()
        return self.head((h * w).sum(1) / w.sum(1).clamp(min=1))

in_dim = toks[int(ridx[0])].shape[1]
torch.manual_seed(0)
model = TunedTransformer(in_dim, D_MODEL, NHEAD, LAYERS, DROPOUT).to(DEVICE)
sum(p.numel() for p in model.parameters())

346177

## Train with LR schedule + early stopping

AdamW + cosine LR decay. Each epoch logs **train loss, val loss, and learning rate**; we keep the **best** weights (marked `*best`) and stop if val does not improve for `PATIENCE` epochs.

In [5]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
dl_tr, dl_va = loader(ttr, True), loader(rva, False)

def val_loss():
    model.eval(); tot = 0.0; nb = 0
    with torch.no_grad():
        for X, m, yb in dl_va:
            tot += nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE)).item(); nb += 1
    return tot / max(nb, 1)

best = float("inf"); best_state = None; wait = 0; best_ep = 0
train_hist, val_hist, lr_hist = [], [], []
for ep in range(EPOCHS):
    model.train(); tot = 0.0; nb = 0
    for X, m, yb in dl_tr:
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(X.to(DEVICE), m.to(DEVICE)), yb.to(DEVICE))
        loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    tr = tot / max(nb, 1); v = val_loss(); lr_now = sched.get_last_lr()[0]
    sched.step()
    train_hist.append(tr); val_hist.append(v); lr_hist.append(lr_now)
    tag = ""
    if v < best - 1e-4:
        best = v; best_state = copy.deepcopy(model.state_dict()); wait = 0; best_ep = ep + 1; tag = "  *best"
    else:
        wait += 1
    print(f"epoch {ep + 1:3d}  train {tr:.4f}  val {v:.4f}  lr {lr_now:.2e}{tag}", flush=True)
    if wait >= PATIENCE:
        print(f"early stop at epoch {ep + 1} (val flat for {PATIENCE})", flush=True)
        break
model.load_state_dict(best_state)
{"epochs_run": ep + 1, "best_epoch": best_ep, "best_val_loss": round(best, 4)}

epoch   1  train 0.5614  val 0.0819  lr 3.00e-04  *best


epoch   2  train 0.1533  val 0.0748  lr 3.00e-04  *best


epoch   3  train 0.1405  val 0.0794  lr 3.00e-04


epoch   4  train 0.1341  val 0.0683  lr 3.00e-04  *best


epoch   5  train 0.1318  val 0.0756  lr 2.99e-04


...

epoch  32  train 0.1151  val 0.0706  lr 2.69e-04


epoch  33  train 0.1149  val 0.0663  lr 2.68e-04


epoch  34  train 0.1146  val 0.0698  lr 2.66e-04


early stop at epoch 34 (val flat for 20)


{'epochs_run': 34, 'best_epoch': 14, 'best_val_loss': 0.064}

## Training curves

Train and validation loss per epoch (MSE on `log E`). The dashed line marks the best epoch that early-stopping kept.

In [6]:
import plotly.graph_objects as go
ep_x = list(range(1, len(train_hist) + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=ep_x, y=train_hist, mode="lines", name="train loss", line=dict(color="#4c78a8")))
fig.add_trace(go.Scatter(x=ep_x, y=val_hist, mode="lines", name="val loss", line=dict(color="#f58518")))
fig.add_vline(x=best_ep, line_dash="dash", line_color="green", annotation_text=f"best (epoch {best_ep})")
fig.update_layout(title="Training and validation loss (MSE on log E)", xaxis_title="epoch",
                  yaxis_title="loss", template="plotly_white", height=430, width=760)
fig

## Predict, then calibrate the output

The raw transformer can have a small scale/offset bias. We fit an affine map on the **validation** predictions (`y ~ a*pred + b`) and apply it to test — same trick the baselines use. This removes systematic bias without touching the test set.

In [7]:
def predict(idx):
    model.eval(); out = []
    with torch.no_grad():
        for X, m, _ in loader(idx, False):
            out.append(model(X.to(DEVICE), m.to(DEVICE)).cpu().numpy().ravel())
    return np.concatenate(out)

pv, pt = predict(rva), predict(rte)
a, b = np.polyfit(pv, y[rva], 1)
raw = resolution(np.exp(pt), Et[rte])
cal = resolution(np.exp(a * pt + b), Et[rte])
{"raw": raw, "calibrated": cal}

{'raw': {'sigma_eff': 0.0509, 'iqr': 0.0504, 'bias': 0.0339},
 'calibrated': {'sigma_eff': 0.0465, 'iqr': 0.0436, 'bias': 0.0165}}

## The bar — same R3 test set

Recompute the competitors on the identical test set for a fair comparison: BDT on the 6 aggregate features, and LHCb `total_energy` (calibrated).

In [8]:
gb = HistGradientBoostingRegressor(max_iter=300, random_state=0)
gb.fit(D["agg"][rtr], y[rtr])
bdt = resolution(np.exp(gb.predict(D["agg"][rte])), Et[rte])

ca, cb = np.polyfit(np.log(D["total_energy"][rtr] + EPS), y[rtr], 1)
te = resolution(np.exp(ca * np.log(D["total_energy"][rte] + EPS) + cb), Et[rte])
{"BDT": bdt, "total_energy": te}

{'BDT': {'sigma_eff': 0.0563, 'iqr': 0.0518, 'bias': 0.0099},
 'total_energy': {'sigma_eff': 0.0579, 'iqr': 0.0593, 'bias': 0.0184}}

## Verdict

Lower `sigma_eff` wins. If the calibrated transformer is below the bar, we convert this notebook to a script and scale up.

In [9]:
bar = min(bdt["sigma_eff"], te["sigma_eff"])
trans = cal["sigma_eff"]
{"transformer_calibrated": trans, "bar": round(bar, 4),
 "delta_vs_bar": round(trans - bar, 4),
 "verdict": "TRANSFORMER WINS" if trans < bar else "not yet - bar still lower"}

{'transformer_calibrated': 0.0465,
 'bar': 0.0563,
 'delta_vs_bar': -0.0098,
 'verdict': 'TRANSFORMER WINS'}

## Prediction quality on the R3 test set (interactive)

Three interactive Plotly views of the trained model on the held-out test set: predicted vs true energy (hug the diagonal), the residual distribution (narrow, centred on zero), and resolution vs energy (the calorimeter a/sqrt(E) curve). Hover for values.

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

Epred = np.exp(a * pt + b)
Etrue = Et[rte]
resid = (Epred - Etrue) / Etrue

def sig_eff(x):
    xs = np.sort(x); n = len(xs); k = max(1, int(np.ceil(0.683 * n)))
    w = xs[k - 1:] - xs[:n - k + 1]
    return float(w.min() / 2)

lim = [float(Etrue.min()), float(Etrue.max())]
fig = go.Figure()
fig.add_trace(go.Scatter(x=Etrue, y=Epred, mode="markers",
                         marker=dict(size=4, opacity=0.3, color="#4c78a8"), name="clusters"))
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(color="crimson", dash="dash"), name="perfect"))
fig.update_layout(title="Predicted vs true energy (R3 test)", xaxis_title="true E [GeV]",
                  yaxis_title="predicted E [GeV]", xaxis_type="log", yaxis_type="log",
                  template="plotly_white", height=460, width=620)
fig

In [11]:
se = sig_eff(resid); med = float(np.median(resid))
fig = go.Figure(go.Histogram(x=resid, xbins=dict(start=-0.3, end=0.3, size=0.0075), marker_color="#72b7b2"))
fig.add_vline(x=med, line_color="crimson", annotation_text=f"bias {med:+.3f}", annotation_position="top")
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(title=f"Residual distribution (sigma_eff = {se:.3f})",
                  xaxis_title="(E_pred - E_true) / E_true", yaxis_title="clusters",
                  template="plotly_white", height=420, width=680)
fig

In [12]:
edges = np.quantile(Etrue, np.linspace(0, 1, 9))
cx, cy = [], []
for i in range(len(edges) - 1):
    msk = (Etrue >= edges[i]) & (Etrue < edges[i + 1])
    if msk.sum() > 20:
        cx.append(float(np.median(Etrue[msk]))); cy.append(sig_eff(resid[msk]))
fig = go.Figure(go.Scatter(x=cx, y=cy, mode="lines+markers", marker=dict(size=9, color="#54a24b")))
fig.update_layout(title="Resolution vs energy", xaxis_title="true E [GeV]", yaxis_title="sigma_eff",
                  xaxis_type="log", template="plotly_white", height=420, width=680)
fig

## Results — tuned transformer vs the bar (interactive)

In [13]:
labels = ["tuned transformer", "BDT", "LHCb total_energy"]
vals = [cal["sigma_eff"], bdt["sigma_eff"], te["sigma_eff"]]
fig = go.Figure(go.Bar(x=vals, y=labels, orientation="h",
                       marker_color=["#2ca02c", "#8c8c8c", "#8c8c8c"],
                       text=[f"{v:.3f}" for v in vals], textposition="outside"))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="R3 test resolution: tuned transformer vs the bar",
                  xaxis_title="sigma_eff  (lower is better)", template="plotly_white",
                  height=320, width=720, margin=dict(l=160))
fig

## Predictions on individual test clusters

Five test clusters spanning the energy range. The table gives true energy, predicted energy, and the residual (bias) per cluster; the panels below show the cells around each seed (marker size and colour = cell energy, seed at the origin).

In [14]:
import pandas as pd
order = np.argsort(Etrue)
picks = order[np.linspace(0, len(order) - 1, 5).astype(int)]
pd.DataFrame({"test_cluster": picks.tolist(),
              "true_E_GeV": np.round(Etrue[picks], 2),
              "pred_E_GeV": np.round(Epred[picks], 2),
              "residual": np.round(resid[picks], 3)})

,test_cluster,true_E_GeV,pred_E_GeV,residual
0,991,2.22,4.69,1.116
1,1373,10.17,10.35,0.018
2,1111,16.80,17.72,0.054
3,1257,24.76,24.70,-0.002
4,1292,156.08,7.05,-0.955


In [15]:
gidx = rte[picks]
titles = [f"true {Etrue[p]:.1f} / pred {Epred[p]:.1f} GeV" for p in picks]
fig = make_subplots(rows=1, cols=5, subplot_titles=titles, horizontal_spacing=0.03)
for col, gi in enumerate(gidx, start=1):
    e, fr, bk, rx, ry, rdr, pitch, mod, cx, cy = D["raw"][int(gi)]
    ps = pitch[int(np.argmax(e))]
    fig.add_trace(go.Scatter(x=rx / ps, y=ry / ps, mode="markers",
        marker=dict(size=10 + 34 * e / e.max(), color=np.log1p(e), colorscale="Viridis",
                    showscale=(col == 5), colorbar=dict(title="log E") if col == 5 else None,
                    line=dict(width=1, color="#333")),
        hovertext=[f"E={ee:.0f} MeV" for ee in e], hoverinfo="text", showlegend=False), row=1, col=col)
fig.update_xaxes(title_text="rel_x/pitch"); fig.update_yaxes(title_text="rel_y/pitch")
fig.update_layout(title="5 example test clusters (cells around seed; size/colour = energy)",
                  template="plotly_white", height=360, width=1500)
fig

## Multi-seed verdict and ablation (from the scripts)

The single run above wobbles run-to-run. These are the 5-seed means from `run_tuned.py` / `run_tuned_ablation.py`: **A** is the winner (tuned transformer, all-region); **B** isolates whether all-region data matters; **C** isolates whether attention matters.

In [16]:
import json
import pandas as pd

rp = repo / "reports"
A = json.loads((rp / "tuned_3x3_allregion.json").read_text())["tuned_transformer"]
abl = json.loads((rp / "tuned_ablation.json").read_text())
rows = [{"config": "A: tuned transformer, all-region", "mean_sigma_eff": A["mean"], "std": A["std"]}]
for k, v in abl["ablation"].items():
    rows.append({"config": k, "mean_sigma_eff": v["mean"], "std": v["std"]})
rows.append({"config": "BDT bar", "mean_sigma_eff": abl["bar_BDT"], "std": 0.0})
pd.DataFrame(rows).sort_values("mean_sigma_eff").reset_index(drop=True)

,config,mean_sigma_eff,std
0,"A: tuned transformer, all-region",0.0453,0.0010
1,A_transformer_allregion,0.0453,0.0010
2,B_transformer_r3only,0.0555,0.0021
3,BDT bar,0.0563,0.0000
4,C_deepsets_allregion,0.0703,0.0032
